# Create lease
1. Choose site
2. Select a project that you are member of
3. Specify a lease name that you are not already using in this site
4. Choose the desired properties (e.g., "compute_skylake") of the baremetal machine
5. Verify that the lease was successful

In [ ]:
import chi
chi.use_site('CHI@TACC')

PROJECT_name = "CHI-261566"
chi.set("project_name", PROJECT_name)
print(f'Using Project {chi.get("project_name")}')

In [ ]:
from chi import lease

import time
import keystoneauth1, blazarclient
from chi import lease

reservations = []
LEASE_KEY = 'reproduce-vldb24-bptree'
lease_node_type = "compute_skylake"
try:
    lease_obj=lease.get_lease(LEASE_KEY)
except Exception as e:
    lease_obj=None
if lease_obj!=None:
    print("lease already created")
    print(lease_obj.node_reservations[0]["id"])
    # print(lease_obj.node_reservations["id"])
else:
    try:
        print(f"Creating lease with name = {LEASE_KEY}...")
        lease.add_fip_reservation(reservations, count=1)
        lease.add_node_reservation(reservations, count=1, node_type=lease_node_type)

        start_date, end_date = lease.lease_duration(hours=6, days=0)

        l = lease.create_lease(
            LEASE_KEY, 
            reservations, 
            start_date=start_date, 
            end_date=end_date
        )
        # l = lease.create_lease(
        #     LEASE_KEY, 
        #     reservations
        # )
        # print(l)
        lease_id = l["id"]
        lease_obj=lease.get_lease(LEASE_KEY)

        print("Waiting for lease to start ...")
        lease_obj.wait()
        print("Lease is now active!")
    #except keystoneauth1.exceptions.http.Unauthorized as e:
        #print("Unauthorized.\nDid set your project name and and run the code in the first cell?")
    except blazarclient.exception.BlazarClientException as e:
        print(f"There is an issue making the reservation. Please check the host calendar.")
        print("https://chi.uc.chameleoncloud.org/project/leases/calendar/host/")
        print(e)
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

In [ ]:
#lease_obj
print(lease_obj.node_reservations[0]["id"])
print(lease_obj.node_reservations[0])
print(lease_obj)

# Create server and assign ip

In [ ]:
from chi import server
server.list_servers()

image = "CC-Ubuntu24.04"
LEASE_KEY = 'reproduce-vldb24-bptree'
try:
    s=server.get_server(LEASE_KEY)
except Exception as e:
    s=None

if s!=None:
    print("server already started")
else:
    try:
        s = server.create_server(
            LEASE_KEY, 
            image_name=image,
            reservation_id=lease_obj.node_reservations[0]["id"]
        )
    except Exception as e:
        print("An unexpected error happened.")
        print(e)

    print("Waiting for server to start ...")
    server.wait_for_active(s.id)
    print("Done")

In [ ]:
server.get_server('reproduce-vldb24-bptree').status

In [ ]:
import chi
from chi import lease

server_id = chi.server.get_server_id(LEASE_KEY)
print(server_id)

floating_ip = lease_obj.get_reserved_floating_ips()
print(floating_ip)
floating_ip = floating_ip[0]
try:
    ip=server.get_floating_ip()
except Exception as e:
    ip=None
if ip != None:
    print("server already get the ip")
else:
    server.associate_floating_ip(server_id, floating_ip_address=floating_ip)
ip = chi.server.get_server(LEASE_KEY).get_floating_ip()
print(ip)

In [ ]:
print(f"Server is accessible at IP address: {ip}")
chi.server.wait_for_tcp(ip, port=22) # Wait for SSH to be ready

# Create SSH connection
All commands are passed through this connection

In [ ]:
import chi
from chi.ssh import Remote
import paramiko

key_path = "/work/.ssh/id_rsa"

# Load your private key
pkey = paramiko.RSAKey.from_private_key_file(key_path)

conn = Remote(
    ip=ip,
    user="cc",
    connect_kwargs={
        "pkey": pkey,
        "key_filename": [],   # change from None → [] so Fabric doesn't crash
        # optional, but often good to be explicit:
        "allow_agent": False,
        "look_for_keys": False,
    },
)

print(conn.run("hostname"))
conn.run("hostname")

In [ ]:
conn.run('ls -al')

In [ ]:
conn.run('sudo apt update')

# Check properties of the newly created instance

In [ ]:
conn.run('free -h')
conn.run('df -h')
conn.run('lsblk')

# Clone the repository for experiment reproduciblity (forked from the original)

In [ ]:
conn.run('git clone https://github.com/S-Saqib/trovi_reproduce_bptree-vldb24-fork.git')

In [ ]:
conn.run('ls -al')

# There is already a script for automating the reproduction of some result
Installs dependencies
Compiles & runs
Generate a figure with the result (figure 5 of the paper)

In [ ]:
conn.run('chmod 777 trovi_reproduce_bptree-vldb24-fork/automate.sh')
conn.run('cd trovi_reproduce_bptree-vldb24-fork && ./automate.sh')

# Get the figure so that it can be viewed in the UI
A python script has already generated the figure named

In [ ]:
conn.run('ls trovi_reproduce_bptree-vldb24-fork/btree_tests')
conn.run('cd trovi_reproduce_bptree-vldb24-fork/btree_tests && python3 plot_figure5.py')

In [ ]:
conn.get(
    'trovi_reproduce_bptree-vldb24-fork/btree_tests/Figure5.png',
    'Figure5.png'
)

In [ ]:
from IPython.display import Image, display
display(Image(filename='Figure5.png'))